In [23]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_validate
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils import resample

df = pd.read_csv('../winequality-red-clean.csv', delimiter=',')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1458 non-null   float64
 1   volatile acidity      1458 non-null   float64
 2   citric acid           1458 non-null   float64
 3   residual sugar        1458 non-null   float64
 4   chlorides             1458 non-null   float64
 5   free sulfur dioxide   1458 non-null   float64
 6   total sulfur dioxide  1458 non-null   float64
 7   density               1458 non-null   float64
 8   pH                    1458 non-null   float64
 9   sulphates             1458 non-null   float64
 10  alcohol               1458 non-null   float64
 11  quality               1458 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 136.8 KB


In [24]:

from sklearn.model_selection import train_test_split

'''
Selected Features classification:
 Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'], dtype='object')
'''

selected_features = ['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
                     'alcohol']
X_selected = df[selected_features]
y = df['quality']

standard_scaler = StandardScaler()
X_selected = standard_scaler.fit_transform(X_selected)

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

In [25]:
def calculate_bias_variance(model, X, y, num_samples=100):
    # Array to store predictions from all bootstrapped models
    all_predictions = np.zeros((num_samples, X.shape[0]))

    # Perform bootstrapping
    for i in range(num_samples):
        # Resample the training data (bootstrap sample)
        X_resampled, y_resampled = resample(X, y)

        # Train the model on the resampled data
        model.fit(X_resampled, y_resampled)

        # Make predictions on the entire dataset (both train and test data)
        all_predictions[i, :] = model.predict(X)

    # Calculate the mean prediction for each data point
    mean_predictions = np.mean(all_predictions, axis=0)

    # Calculate the bias^2: The squared difference between true values and mean predictions
    bias_squared = np.mean((mean_predictions - y) ** 2)
    bias_percent = (bias_squared / np.mean(y_test)) * 100

    # Calculate the variance: The average variance of predictions across different bootstrapped models
    variance = np.mean(np.var(all_predictions, axis=0))
    variance_percent = (variance / np.mean(y_test)) * 100

    print(f"Bias^2 : {bias_percent:.2f}%")
    print(f"Variance: {variance_percent:.2f}%")
    

In [26]:
def train_test_accuracy(train_accuracy, test_accuracy):
    print(f'Training Accuracy: {(train_accuracy * 100).__round__(3)}%')
    print(f'Testing Accuracy: {(test_accuracy * 100).__round__(3)}%')
    

In [27]:
param_grid = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

# Initialize the OneVsRestClassifier with a linear SVM as the base model
ovr_linear_model = OneVsRestClassifier(SVC(kernel='linear'))

# Initialize GridSearchCV to perform hyperparameter tuning
grid_search = GridSearchCV(estimator=ovr_linear_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1)

# Perform the grid search with cross-validation
grid_search.fit(X_train, y_train)
# Extract the best model and its corresponding parameters
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Output the best parameters and the corresponding accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

# You can also perform cross-validation with the best model to check the bias-variance tradeoff
cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model, X_test, y_test)

'''
Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best Parameters: {'estimator__C': 1000}
Best Cross-Validation Accuracy: 0.5378

'''


Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best Parameters: {'estimator__C': 1000}
Best Cross-Validation Accuracy: 0.5360
OneVsRestClassifier with Best Parameters:
Training Accuracy: 54.938%
Testing Accuracy: 53.776%
Bias^2 : 7.18%
Variance: 1.48%


"\nFitting 5 folds for each of 7 candidates, totalling 35 fits\nBest Parameters: {'estimator__C': 1000}\nBest Cross-Validation Accuracy: 0.5378\n\n"

In [28]:
# Define the parameter grid for hyperparameter tuning
param_grid_rbf = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],  # Regularization parameter
    'estimator__gamma': [0.001, 0.01, 0.1, 1, 10]  # Gamma parameter for the RBF kernel
}

# Initialize the OneVsRestClassifier with an SVM using the RBF kernel as the base model
ovr_rbf_model = OneVsRestClassifier(SVC(kernel='rbf'))

# Initialize GridSearchCV to perform hyperparameter tuning
grid_search = GridSearchCV(estimator=ovr_rbf_model, param_grid=param_grid_rbf, cv=5, scoring='accuracy', verbose=1)

# Perform the grid search with cross-validation
grid_search.fit(X_train, y_train)

# Extract the best model and its corresponding parameters
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Output the best parameters and the corresponding accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

# You can also perform cross-validation with the best model to check the bias-variance tradeoff
cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier RBF with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model, X_test, y_test)
'''
Best Parameters: {'estimator__C': 1, 'estimator__gamma': 0.1}
'''

Fitting 5 folds for each of 35 candidates, totalling 175 fits
Best Parameters: {'estimator__C': 10, 'estimator__gamma': 1}
Best Cross-Validation Accuracy: 0.6141
OneVsRestClassifier RBF with Best Parameters:
Training Accuracy: 91.581%
Testing Accuracy: 50.823%
Bias^2 : 1.32%
Variance: 3.32%


"\nBest Parameters: {'estimator__C': 1, 'estimator__gamma': 0.1}\n"

In [29]:
# Define the parameter grid for hyperparameter tuning (varying C values)
param_grid = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]  # Values for the regularization parameter C
}

# Initialize the OneVsOneClassifier with a linear SVM as the base model
ovo_linear_model = OneVsOneClassifier(SVC(kernel='linear'))

# Initialize GridSearchCV to perform hyperparameter tuning
grid_search = GridSearchCV(estimator=ovo_linear_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1)

# Perform the grid search with cross-validation
grid_search.fit(X_train, y_train)

# Extract the best model and its corresponding parameters
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Output the best parameters and the corresponding accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

# You can also perform cross-validation with the best model to check the bias-variance tradeoff
cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model, X_test, y_test)
'''
Best Parameters: {'estimator__C': 10}
'''

Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best Parameters: {'estimator__C': 0.1}
Best Cross-Validation Accuracy: 0.5712
OneVsOneClassifier with Best Parameters:
Training Accuracy: 58.196%
Testing Accuracy: 57.136%
Bias^2 : 7.61%
Variance: 0.96%


"\nBest Parameters: {'estimator__C': 10}\n"

In [30]:
# Define the parameter grid for hyperparameter tuning
param_grid_rbf = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],  # Regularization parameter
    'estimator__gamma': [0.001, 0.01, 0.1, 1, 10]  # Gamma parameter for the RBF kernel
}

# Initialize the OneVsOneClassifier with an SVM using the RBF kernel as the base model
ovo_rbf_model = OneVsOneClassifier(SVC(kernel='rbf'))

# Initialize GridSearchCV to perform hyperparameter tuning
grid_search = GridSearchCV(estimator=ovo_rbf_model, param_grid=param_grid_rbf, cv=5, scoring='accuracy', verbose=1)

# Perform the grid search with cross-validation
grid_search.fit(X_train, y_train)

# Extract the best model and its corresponding parameters
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Output the best parameters and the corresponding accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

# You can also perform cross-validation with the best model to check the bias-variance tradeoff
cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier RBF with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model, X_test, y_test)
'''
Best Parameters: {'estimator__C': 1, 'estimator__gamma': 0.1}
'''

Fitting 5 folds for each of 35 candidates, totalling 175 fits
Best Parameters: {'estimator__C': 1000, 'estimator__gamma': 1}
Best Cross-Validation Accuracy: 0.6132
OneVsOneClassifier RBF with Best Parameters:
Training Accuracy: 100.0%
Testing Accuracy: 47.39%
Bias^2 : 1.21%
Variance: 3.13%


"\nBest Parameters: {'estimator__C': 1, 'estimator__gamma': 0.1}\n"